# 📓 Notebook 03 — Embeddings & Semantic Search---**Phase 1 · LLM Fundamentals**> Embeddings are the **backbone of RAG, search, recommendations, and clustering**. Master them and you unlock everything.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Vector spaces | How meaning becomes math || Embedding models | Multi-model: OpenAI, Gemini, sentence-transformers || Distance metrics | Cosine vs dot product vs euclidean — when to use which || Semantic search | Find by meaning, not keywords || Practical patterns | Batching, caching, dimensionality |### 🏗️ Build: Multi-Model Semantic Search Engine

## 1. Setup

In [ ]:
import os, json, timeimport numpy as npfrom dotenv import load_dotenvimport litellmload_dotenv()DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")DEFAULT_EMBEDDING = os.getenv("DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small")print(f"🤖 LLM: {DEFAULT_MODEL}")print(f"📐 Embedding: {DEFAULT_EMBEDDING}")

---## 2. What Are Embeddings?### The Core IdeaAn **embedding** is a dense vector (list of floats) that represents the **meaning** of text in a high-dimensional space.```"The cat sat on the mat"  →  [0.023, -0.156, 0.891, ..., 0.042]   # 1536 dimensions```### Why This Matters- **Similar meanings → nearby vectors** → enables semantic search- **Different from keywords** → "automobile" ≈ "car" even though different words- Powers: search, RAG, recommendations, clustering, anomaly detection### How Embeddings Work (Interview Depth)```                     Text Input                         ↓               ┌─────────────────┐               │  Transformer    │   (BERT, GPT, etc.)               │  Encoder        │               └────────┬────────┘                         ↓              [0.02, -0.15, 0.89, ...]   ← Dense vector (1536-D)                         ↓               Meaning in vector space```| Property | Detail ||----------|--------|| Dimensions | 256–3072 depending on model || Values | Floating point numbers, typically -1 to 1 (normalized) || Key insight | Cosine similarity between vectors ≈ semantic similarity || Training | Learned from massive text corpora (contrastive learning) |

In [ ]:
def get_embedding(text, model=None):    """Get embedding vector for text using litellm."""    model = model or DEFAULT_EMBEDDING    response = litellm.embedding(model=model, input=[text])    return response.data[0]["embedding"]def get_embeddings_batch(texts, model=None):    """Batch embed multiple texts (more efficient)."""    model = model or DEFAULT_EMBEDDING    response = litellm.embedding(model=model, input=texts)    return [d["embedding"] for d in response.data]# Get our first embeddingtext = "Machine learning is a subset of artificial intelligence."embedding = get_embedding(text)print(f"📐 Embedding for: \"{text}\"")print(f"   Dimensions: {len(embedding)}")print(f"   First 10 values: {[round(v, 4) for v in embedding[:10]]}")print(f"   Min: {min(embedding):.4f}, Max: {max(embedding):.4f}")print(f"   Mean: {np.mean(embedding):.4f}, Std: {np.std(embedding):.4f}")print(f"   L2 Norm: {np.linalg.norm(embedding):.4f}")

---## 3. Distance Metrics — Measuring Similarity### The Three Main Metrics| Metric | Formula | Range | Best For ||--------|---------|-------|----------|| **Cosine Similarity** | cos(θ) = A·B / (\|A\|·\|B\|) | [-1, 1] | Normalized embeddings (most common) || **Dot Product** | A·B | (-∞, +∞) | When magnitude matters (popularity) || **Euclidean Distance** | √Σ(aᵢ-bᵢ)² | [0, +∞) | When absolute distance matters |> **Interview Critical:** Most embedding models output *normalized* vectors (L2 norm ≈ 1).  > For normalized vectors, cosine similarity = dot product. This is why FAISS uses inner product by default.

In [ ]:
def cosine_similarity(a, b):    """Cosine similarity between two vectors."""    a, b = np.array(a), np.array(b)    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))def dot_product(a, b):    """Dot product between two vectors."""    return np.dot(np.array(a), np.array(b))def euclidean_distance(a, b):    """Euclidean distance between two vectors."""    return np.linalg.norm(np.array(a) - np.array(b))# Compare semantically similar vs different textspairs = [    ("The cat sat on the mat", "A feline rested on a rug"),            # Same meaning    ("The cat sat on the mat", "The stock market crashed today"),       # Very different    ("Python is a programming language", "Python is a type of snake"), # Ambiguous!    ("I love machine learning", "I enjoy AI and deep learning"),       # Similar domain    ("The weather is sunny today", "It's a beautiful bright day"),     # Paraphrase]print("📊 Similarity Analysis")print("=" * 80)# Batch embed all unique textsall_texts = list(set([t for pair in pairs for t in pair]))all_embeddings = get_embeddings_batch(all_texts)embed_map = dict(zip(all_texts, all_embeddings))for text_a, text_b in pairs:    emb_a = embed_map[text_a]    emb_b = embed_map[text_b]    cos = cosine_similarity(emb_a, emb_b)    dot = dot_product(emb_a, emb_b)    euc = euclidean_distance(emb_a, emb_b)        bar = "█" * int(cos * 30) if cos > 0 else ""    print(f"\n  \"{text_a}\"")    print(f"  \"{text_b}\"")    print(f"  Cosine: {cos:.4f} {bar}")    print(f"  Dot:    {dot:.4f}")    print(f"  Eucl:   {euc:.4f}")

### 💡 When to Use Which Metric| Scenario | Best Metric | Why ||----------|-------------|-----|| Semantic text search | Cosine similarity | Direction matters, not magnitude || Document with popularity scores | Dot product | Longer docs = higher magnitude = boosted || Anomaly detection | Euclidean distance | Absolute distance from cluster center || Normalized vectors (most APIs) | Cosine ≡ Dot product | They're identical when ‖v‖=1 |> **Interview Tip:** *"Are cosine similarity and dot product ever the same?"*  > Yes! When vectors are L2-normalized (unit vectors), cosine = dot product. Most embedding APIs return normalized vectors.

---## 4. Embedding Models — Comparison| Model | Provider | Dimensions | Speed | Quality | Cost ||-------|----------|------------|-------|---------|------|| `text-embedding-3-small` | OpenAI | 1536 | Fast | Good | $0.02/1M tokens || `text-embedding-3-large` | OpenAI | 3072 | Medium | Best | $0.13/1M tokens || `text-embedding-004` | Google | 768 | Fast | Good | Free tier available || `all-MiniLM-L6-v2` | HuggingFace | 384 | Very fast | Good | Free (local) || `all-mpnet-base-v2` | HuggingFace | 768 | Medium | Better | Free (local) |> **Interview Tip:** *"How do you choose an embedding model?"*  > Consider: quality needs, latency requirements, cost budget, deployment constraints (cloud vs local).  > Start with `text-embedding-3-small` for most use cases. Use local models for privacy/offline.

In [ ]:
# Compare embedding dimensions and norms across models# (Uses your default model — you can change DEFAULT_EMBEDDING in .env)test_text = "Artificial intelligence is transforming the way we build software."emb = get_embedding(test_text)print(f"📐 Embedding Model Analysis: {DEFAULT_EMBEDDING}")print(f"   Dimensions: {len(emb)}")print(f"   L2 Norm: {np.linalg.norm(emb):.6f}")print(f"   Is normalized: {'✅ Yes' if abs(np.linalg.norm(emb) - 1.0) < 0.01 else '❌ No'}")print(f"   Memory per vector: {len(emb) * 4} bytes (float32)")print(f"   Memory for 1M vectors: {len(emb) * 4 * 1_000_000 / (1024**3):.2f} GB")

---## 5. Practical Patterns### 5a. Batch ProcessingAlways embed in batches — single calls are 10-50x slower per item.

In [ ]:
# Batch vs single embedding performance comparisontexts = [    "Machine learning automates data analysis",    "Deep learning uses neural networks",     "Natural language processing handles text",    "Computer vision processes images",    "Reinforcement learning trains through rewards",    "Transfer learning reuses pretrained models",    "Federated learning trains across devices",    "AutoML automates model selection",]# Method 1: One at a time (slow)start = time.time()single_embs = [get_embedding(t) for t in texts]single_time = time.time() - start# Method 2: Batch (fast)start = time.time()batch_embs = get_embeddings_batch(texts)batch_time = time.time() - startprint(f"⏱️ Performance Comparison ({len(texts)} texts)")print(f"   Single calls: {single_time:.2f}s ({single_time/len(texts)*1000:.0f}ms per text)")print(f"   Batch call:   {batch_time:.2f}s ({batch_time/len(texts)*1000:.0f}ms per text)")print(f"   Speedup:      {single_time/batch_time:.1f}x faster with batching")

### 5b. Embedding VisualizationHigh-dimensional vectors can be reduced to 2D/3D using **t-SNE** or **PCA** for visualization.

In [ ]:
from sklearn.decomposition import PCAimport numpy as np# Embed categorized textscategories = {    "🐱 Animals": ["The cat hunts mice", "Dogs are loyal pets", "Birds fly south in winter", "Elephants never forget"],    "💻 Tech": ["Neural networks learn patterns", "Python is popular for AI", "GPUs accelerate training", "Cloud computing scales easily"],    "🍕 Food": ["Pizza is popular worldwide", "Sushi requires fresh fish", "Pasta comes from Italy", "Tacos are Mexican cuisine"],}all_texts = []all_labels = []for cat, texts in categories.items():    all_texts.extend(texts)    all_labels.extend([cat] * len(texts))embeddings = get_embeddings_batch(all_texts)emb_matrix = np.array(embeddings)# Reduce to 2D with PCApca = PCA(n_components=2)coords = pca.fit_transform(emb_matrix)print("📊 2D Embedding Visualization (PCA)")print(f"   Explained variance: {pca.explained_variance_ratio_.sum():.1%}")print("=" * 60)# Text-based scatter plotfor i, (label, text) in enumerate(zip(all_labels, all_texts)):    x, y = coords[i]    print(f"  {label} ({x:+.2f}, {y:+.2f}) → \"{text}\"")# Show within-cluster vs between-cluster distancesprint("\n📏 Cluster Distances:")for cat_a in categories:    for cat_b in categories:        if cat_a >= cat_b:            continue        idxs_a = [i for i, l in enumerate(all_labels) if l == cat_a]        idxs_b = [i for i, l in enumerate(all_labels) if l == cat_b]        dists = [cosine_similarity(embeddings[a], embeddings[b]) for a in idxs_a for b in idxs_b]        print(f"  {cat_a} ↔ {cat_b}: avg cosine = {np.mean(dists):.4f}")

---## 📝 Interview Quiz — Embeddings### Q1: What is an embedding? How is it different from one-hot encoding?<details><summary>💡 Answer</summary>**Embedding:** A dense, learned vector representation where similar items are close in vector space.- Dense: Every dimension has a meaningful value- Learned: Trained from data (not manually designed)- Fixed size regardless of vocabulary**One-hot encoding:** A sparse binary vector where one position is 1.- Sparse: Mostly zeros- No similarity: every pair is equidistant - Size = vocabulary size (can be millions)**Key difference:** Embeddings capture *semantic similarity* — "king" and "queen" are close, but one-hot treats them as completely unrelated.</details>### Q2: Explain cosine similarity. When does it equal dot product?<details><summary>💡 Answer</summary>**Cosine similarity** = cos(θ) = (A·B) / (‖A‖ × ‖B‖)- Measures the *angle* between two vectors- Range: [-1, 1] (1 = identical direction, 0 = orthogonal, -1 = opposite)- Ignores vector magnitude (length)**Equals dot product when:** vectors are L2-normalized (unit vectors, ‖v‖=1).- Most embedding APIs return normalized vectors- So in practice, cosine similarity = dot product for API embeddings- FAISS `IndexFlatIP` (inner product) is equivalent to cosine search for normalized vectors</details>### Q3: How would you choose an embedding model for production?<details><summary>💡 Answer</summary>Decision factors:1. **Quality**: Check MTEB leaderboard for task-specific benchmarks2. **Latency**: Local models (sentence-transformers) < API models for high QPS3. **Cost**: OpenAI charges per token, local models are free after setup4. **Privacy**: Sensitive data → local models (no data leaves your servers)5. **Dimensions**: More dims = better quality but more storage/compute6. **Context length**: Some models handle longer texts better**Rule of thumb:** Start with `text-embedding-3-small` (OpenAI) or `all-MiniLM-L6-v2` (local), benchmark, then upgrade if needed.</details>### Q4: You have 10 million documents to embed. How would you do it efficiently?<details><summary>💡 Answer</summary>1. **Batch processing**: Send 100-2000 texts per API call2. **Parallel requests**: Use asyncio or thread pools (respect rate limits)3. **Cache results**: Store embeddings alongside documents (avoid re-computation)4. **Chunk large documents**: Most models have token limits (8192 for OpenAI)5. **Use cheaper models**: `text-embedding-3-small` over `large` unless quality demands it6. **Local models**: For high volume, run sentence-transformers on GPU — no API costs7. **Incremental updates**: Only embed new/changed documents, not the entire corpus**Cost estimate:** 10M docs × 500 tokens avg × $0.02/1M tokens = $100 with OpenAI small model.</details>### Q5: What is the "curse of dimensionality"? How does it affect similarity search?<details><summary>💡 Answer</summary>In very high-dimensional spaces (1000+):- All points become roughly **equidistant** from each other- **Nearest neighbor** becomes less meaningful- The difference between nearest and farthest neighbor shrinks**Impact on similarity search:**- Exact nearest-neighbor search becomes slow (must check every vector)- Approximate methods (HNSW, IVF) become necessary- Dimensionality reduction (PCA) can help but loses information**Why embeddings still work:** Even though raw distances converge, the **relative ordering** is preserved. The most similar document is still returned first — the absolute scores just become less interpretable.</details>### Q6: Can you use the same embedding model for different languages?<details><summary>💡 Answer</summary>**Depends on the model:**- **OpenAI's text-embedding-3**: Multilingual, works across 100+ languages in the same vector space- **sentence-transformers/multilingual models**: Specifically trained for cross-lingual similarity- **English-only models**: Will produce poor results for other languages**Cross-lingual search:** Multilingual models let you query in English and find Hindi documents — the embeddings share the same semantic space regardless of language.**Best practice:** Test with your specific languages before committing. Use multilingual benchmarks (MTEB multilingual subset).</details>

---## 🏗️ BUILD: Multi-Model Semantic Search EngineA complete in-memory semantic search engine with:- Multi-model embedding support- Cosine similarity ranking- Result explanation- Performance metrics

In [ ]:
class SemanticSearchEngine:    """In-memory semantic search engine with multi-model support."""        def __init__(self, embedding_model=None):        self.model = embedding_model or DEFAULT_EMBEDDING        self.documents = []      # Original texts        self.metadata = []       # Optional metadata per doc        self.embeddings = []     # Embedding vectors        self.index_built = False        def add_documents(self, texts, metadata_list=None):        """Add documents to the search index."""        # Batch embed        new_embeddings = get_embeddings_batch(texts, model=self.model)                self.documents.extend(texts)        self.embeddings.extend(new_embeddings)        if metadata_list:            self.metadata.extend(metadata_list)        else:            self.metadata.extend([{} for _ in texts])                self.index_built = True        print(f"  📥 Added {len(texts)} documents (total: {len(self.documents)})")        def search(self, query, top_k=5):        """Search for documents most similar to the query."""        assert self.index_built, "No documents indexed yet!"                # Embed query        query_emb = get_embedding(query, model=self.model)                # Calculate similarities        similarities = []        for i, doc_emb in enumerate(self.embeddings):            sim = cosine_similarity(query_emb, doc_emb)            similarities.append((i, sim))                # Sort by similarity (descending)        similarities.sort(key=lambda x: x[1], reverse=True)                # Return top_k results        results = []        for idx, sim in similarities[:top_k]:            results.append({                "rank": len(results) + 1,                "text": self.documents[idx],                "score": round(sim, 4),                "metadata": self.metadata[idx],            })        return results        def stats(self):        """Print index statistics."""        if not self.embeddings:            print("  Empty index")            return        emb_array = np.array(self.embeddings)        print(f"  📊 Index Stats")        print(f"     Documents: {len(self.documents)}")        print(f"     Dimensions: {emb_array.shape[1]}")        print(f"     Memory: {emb_array.nbytes / 1024:.1f} KB")        print(f"     Model: {self.model}")# Build the search engineengine = SemanticSearchEngine()# Add a knowledge baseknowledge_base = [    "Python is a high-level programming language known for its readability and versatility.",    "Machine learning algorithms learn patterns from data to make predictions.",    "Docker containers package applications with their dependencies for consistent deployment.",    "REST APIs use HTTP methods like GET, POST, PUT, DELETE for web services.",    "PostgreSQL is an open-source relational database with advanced features.",    "Kubernetes orchestrates containerized applications across clusters of machines.",    "React is a JavaScript library for building user interfaces with components.",    "Git version control tracks changes in source code during development.",    "Neural networks are computing systems inspired by biological brain networks.",    "Redis is an in-memory data store used for caching and message brokering.",    "GraphQL is a query language for APIs that gives clients control over data fetching.",    "TensorFlow and PyTorch are popular frameworks for deep learning research and deployment.",    "Microservices architecture breaks applications into small, independently deployable services.",    "OAuth 2.0 is an authorization framework for secure access delegation.",    "WebSocket provides full-duplex communication channels over a single TCP connection.",    "Natural language processing enables computers to understand and generate human language.",    "Apache Kafka is a distributed event streaming platform for high-throughput data pipelines.",    "HTTPS encrypts web traffic using TLS/SSL certificates for secure communication.",    "Agile methodology emphasizes iterative development, collaboration, and rapid delivery.",    "Vector databases store and search high-dimensional embeddings for AI applications.",]metadata = [{"id": i, "category": "tech"} for i in range(len(knowledge_base))]print("🔨 Building Search Index")engine.add_documents(knowledge_base, metadata)engine.stats()

In [ ]:
# Search the knowledge basequeries = [    "How do I build a web API?",    "What's the best way to deploy my application?",    "I need to store data efficiently",    "How can AI understand text?",    "secure my website",]print("🔍 Semantic Search Results")print("=" * 70)for query in queries:    results = engine.search(query, top_k=3)    print(f"\n🔎 Query: \"{query}\"")    for r in results:        relevance = "🟢" if r["score"] > 0.5 else "🟡" if r["score"] > 0.3 else "🔴"        print(f"   {relevance} [{r['score']:.4f}] {r['text'][:80]}...")

In [ ]:
# Edge case: query in a different language (if using multilingual model)print("🌍 Cross-Domain Search Test")print("=" * 50)edge_queries = [    "containerization and devops",     # Should match Docker/K8s    "frontend development framework",  # Should match React    "data pipeline streaming",         # Should match Kafka    "authentication and security",     # Should match OAuth/HTTPS]for q in edge_queries:    results = engine.search(q, top_k=2)    print(f"\n🔎 \"{q}\"")    for r in results:        print(f"   [{r['score']:.4f}] {r['text'][:70]}...")

---## ✅ Notebook 03 Summary| Concept | Key Takeaway ||---------|-------------|| Embeddings | Dense vectors capturing semantic meaning; similar text → nearby vectors || Cosine similarity | Most common metric; range [-1, 1]; direction-based || Dot product | = Cosine for normalized vectors; used by FAISS || Euclidean | Absolute distance; good for anomaly detection || Batch embedding | 10-50x faster than single calls; always batch || Model choice | OpenAI small for general; local (sentence-transformers) for privacy/cost || Curse of dimensionality | Distances converge in high-D but relative ordering is preserved |### ➡️ Next: [Notebook 04 — Basic RAG](./04_basic_rag.ipynb)*Combine embeddings with LLMs to build your first Retrieval-Augmented Generation pipeline.*